[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/08_From_Gradient_Descent_to_Boosted_Trees.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/foundations/08_From_Gradient_Descent_to_Boosted_Trees.ipynb)

# From Gradient Descent to Gradient Boosted Trees

Gradient descent is a general learning algorithm: if you can compute an error and its derivative, you can learn.

This notebook deepens the *why* behind each piece of gradient descent, then shows how the same idea powers **Gradient Boosted Trees**.

---

**What we cover**

1. The derivative — why `weight_delta = input × delta` is the slope of the loss
2. Divergence — what happens when alpha or input is too large
3. The frozen weight — what happens when a feature value is zero
4. The error plane — how training data shapes the loss surface
5. GD in function space — how boosted trees apply gradient descent one tree at a time


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.dpi'] = 110


## Part 1 — The Derivative: Why `weight_delta = input × delta`

The gradient descent update rule is:

```
delta        = pred - goal          # how wrong the output is
weight_delta = input * delta        # derivative of error w.r.t. this weight
weight      -= alpha * weight_delta # move opposite the slope
```

**Why does multiplying by `input` give the derivative?**

The error formula is:

```
error = (input × weight  −  goal)²
```

Taking the derivative w.r.t. `weight`:

```
d(error)/d(weight) = 2 × (input × weight − goal) × input
                   = 2 × delta × input
```

The `2` is absorbed into `alpha`.  So **`weight_delta = input × delta` IS the slope of the loss curve**.

- Positive slope → weight is too high → subtract → move left toward minimum  
- Negative slope → weight is too low → add → move right toward minimum  
- Steeper slope → bigger correction


In [ ]:
x_input = 0.5
goal    = 0.8

w_range = np.linspace(-0.5, 3.5, 300)
errors  = (w_range * x_input - goal) ** 2

w0      = 0.0
e0      = (w0 * x_input - goal) ** 2
delta0  = w0 * x_input - goal
slope   = 2 * delta0 * x_input
tangent = e0 + slope * (w_range - w0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(w_range, errors, 'steelblue', lw=2.5, label='error = (w*x - goal)^2')
ax.plot(w_range, tangent, '--', color='tomato', lw=1.8, label='slope (derivative) at w=0')
ax.scatter([w0], [e0], color='black', zorder=5, s=80, label='start  w=0')
ax.scatter([goal / x_input], [0], color='green', zorder=5, s=100, marker='*', label='goal  w=1.6')
ax.annotate('move this way ->', xy=(0.25, 0.45), fontsize=9, color='tomato')
ax.set_xlabel('weight value')
ax.set_ylabel('error')
ax.set_title('The loss bowl -- slope gives direction and amount')
ax.set_ylim(-0.1, 0.8)
ax.legend(fontsize=9)

alpha = 0.5
steps = []
w = 0.0
for _ in range(15):
    pred  = w * x_input
    e     = (pred - goal) ** 2
    delta = pred - goal
    steps.append((w, e))
    w -= alpha * (x_input * delta)

ws, es = zip(*steps)
ax2 = axes[1]
ax2.plot(w_range, errors, 'steelblue', lw=2, alpha=0.6)
ax2.scatter(ws, es, color='tomato', zorder=5, s=40)
ax2.plot(ws, es, 'tomato', lw=1.2, alpha=0.8)
ax2.scatter([ws[0]], [es[0]], color='black', s=80, zorder=6, label='start')
ax2.scatter([goal / x_input], [0], color='green', s=100, marker='*', zorder=6, label='goal')
ax2.set_xlabel('weight value')
ax2.set_ylabel('error')
ax2.set_title('GD steps walking down the bowl (alpha=0.5, x=0.5)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"delta        = pred - goal  = {w0*x_input:.2f} - {goal:.2f} = {delta0:.4f}")
print(f"weight_delta = x * delta    = {x_input} x {delta0:.4f} = {x_input*delta0:.4f}")
print(f"slope (true) = 2*delta*x   = {slope:.4f}   (same direction as weight_delta)")
print()
print("weight_delta is the derivative.")
print("Moving opposite it (-= alpha * weight_delta) walks down to the minimum.")


## Part 2 — Divergence: When Gradient Descent Explodes

The update is `weight -= alpha * (input * delta)`.

If `input` is large, `weight_delta` is large — the step overshoots the minimum.
The new error is **larger** than before, causing an even bigger correction next step.
Each correction overcorrects more — the error **diverges** (explodes upward).

**Visualized below**: same goal, same starting weight — only the input size changes.

Two fixes:
- Reduce **alpha** (learning rate)
- **Normalize inputs** (StandardScaler) so no feature dominates the gradient magnitude


In [ ]:
def run_gd(x_val, goal_val, alpha_val, n_iter=25):
    w = 0.5
    errors_out, weights_out = [], []
    for _ in range(n_iter):
        pred  = w * x_val
        e     = (pred - goal_val) ** 2
        delta = pred - goal_val
        errors_out.append(e)
        weights_out.append(w)
        w -= alpha_val * (x_val * delta)
    return weights_out, errors_out

goal = 0.8
configs = [
    (0.5, 0.01, 'x=0.5  alpha=0.01  (converges)',  '#4CAF50'),
    (2.0, 0.01, 'x=2.0  alpha=0.01  (diverges)',   '#E53935'),
    (2.0, 0.001,'x=2.0  alpha=0.001 (fixed)',       '#1E88E5'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for x_val, alpha_val, label, color in configs:
    ws, es = run_gd(x_val, goal, alpha_val, n_iter=25)
    axes[0].plot(range(len(es)), np.clip(es, 0, 2.5), color=color, lw=2, label=label)
    axes[1].plot(ws, color=color, lw=2, label=label)

axes[0].set_xlabel('iteration')
axes[0].set_ylabel('error (clipped at 2.5)')
axes[0].set_title('Divergence: large input makes the loss explode')
axes[0].legend(fontsize=9)
axes[0].axhline(0, color='grey', lw=0.8, ls='--')

axes[1].set_xlabel('iteration')
axes[1].set_ylabel('weight value')
axes[1].set_title('Weight trajectories: converge vs oscillate vs explode')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print("x=0.5, alpha=0.01  -> small input, small gradient, converges smoothly")
print("x=2.0, alpha=0.01  -> large input, large gradient, overshoots both sides, diverges")
print("x=2.0, alpha=0.001 -> same large input, smaller alpha absorbs the scale, converges")
print()
print("This is exactly why StandardScaler is applied before gradient-based models.")
print("Equal feature scales -> equal gradient magnitudes -> stable descent.")


## Part 3 — The Frozen Weight: What Happens When a Feature Value is Zero

Each weight gets its own gradient update:

```
weight_delta[i] = input[i] * delta
```

If `input[i] = 0`, then `weight_delta[i] = 0` — **that weight gets no update from that row**.

If a feature is *always* zero (or always the same value), its weight is never updated
no matter how large the error is.  It is frozen by the data, not by code.

This is the **silent feature rule**: feature = 0 → gradient = 0 → weight frozen.

In the experiment below, we manually force `weight_delta[0] = 0` each step to simulate a
feature that never fires.  The other weights still converge — but the model is weaker.


In [ ]:
np.random.seed(0)
n = 80
X_fw = np.column_stack([
    np.random.uniform(1, 10, n),
    np.random.uniform(0, 1,  n),
    np.random.uniform(0, 1,  n),
])
true_w = np.array([2.0, -3.0, 0.0])
y_fw   = X_fw @ true_w + np.random.normal(0, 0.3, n)

scaler_fw = StandardScaler()
X_fw_sc   = scaler_fw.fit_transform(X_fw)
y_fw_c    = (y_fw - y_fw.mean()) / y_fw.std()

def run_multi_gd(X_data, y_data, freeze_idx=None, lr=0.02, n_iter=300):
    w      = np.zeros(X_data.shape[1])
    b      = 0.0
    losses = []
    wh     = []
    for _ in range(n_iter):
        pred  = X_data @ w + b
        error = pred - y_data
        losses.append((error ** 2).mean())
        wh.append(w.copy())
        gw = (2 / len(y_data)) * (X_data.T @ error)
        gb = (2 / len(y_data)) * error.sum()
        if freeze_idx is not None:
            gw[freeze_idx] = 0.0
        w -= lr * gw
        b -= lr * gb
    return w, losses, np.array(wh)

w_norm, loss_norm, wh_norm = run_multi_gd(X_fw_sc, y_fw_c)
w_froz, loss_froz, wh_froz = run_multi_gd(X_fw_sc, y_fw_c, freeze_idx=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(loss_norm, 'steelblue', lw=2, label='all weights free')
axes[0].plot(loss_froz, 'tomato',    lw=2, label='weight[0] frozen')
axes[0].set_xlabel('iteration')
axes[0].set_ylabel('MSE')
axes[0].set_title('Frozen weight converges to a higher floor')
axes[0].legend()

feat_labels = ['feat0 (informative, frozen)', 'feat1 (informative)', 'feat2 (noise)']
colors_fw   = ['tomato', 'steelblue', 'orange']
for i in range(3):
    axes[1].plot(wh_norm[:, i], color=colors_fw[i], lw=2, label=feat_labels[i])
axes[1].set_title('Normal training -- all weights adapt')
axes[1].set_xlabel('iteration')
axes[1].set_ylabel('weight value')
axes[1].legend(fontsize=8)

for i in range(3):
    ls = '--' if i == 0 else '-'
    axes[2].plot(wh_froz[:, i], color=colors_fw[i], lw=2, ls=ls, label=feat_labels[i])
axes[2].set_title('Frozen weight[0] -- stays 0 despite being informative')
axes[2].set_xlabel('iteration')
axes[2].set_ylabel('weight value')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Normal final weights : {w_norm.round(3)}")
print(f"Frozen final weights : {w_froz.round(3)}")
print(f"Normal final MSE     : {loss_norm[-1]:.4f}")
print(f"Frozen final MSE     : {loss_froz[-1]:.4f}")
print()
print("weight[0] was the most informative feature but was never learned.")
print("The other weights compensated partially -- error is higher than it needs to be.")
print("Real-world equivalent: a feature that is always 0 for a class of observations")
print("will never update its weight for those rows.")


## Part 4 — The Error Plane: How Training Data Shapes the Loss Surface

With one weight, the loss is a bowl (1D).  With two weights, it is an **error plane**: a 3D
surface where height = error, and the two horizontal axes are the two weight values.

The shape is determined entirely by the training data — not the algorithm.

- **Unscaled features** (e.g., age 20–70 vs income 0–1): elongated, narrow valley  
  → GD must take tiny steps to avoid overshooting the steep direction  
- **Scaled features** (StandardScaler): nearly circular bowl  
  → GD can take equal steps in all directions and converges faster

The contour plots below show the same dataset before and after scaling.


In [ ]:
np.random.seed(1)
n_ep  = 100
age   = np.random.uniform(20, 70, n_ep)
inc   = np.random.uniform(0, 1, n_ep)
X_ep  = np.column_stack([age, inc])
y_ep  = 2.0 * age + 50.0 * inc + np.random.normal(0, 5, n_ep)
y_epc = y_ep - y_ep.mean()

scaler_ep = StandardScaler()
X_ep_sc   = scaler_ep.fit_transform(X_ep)

def mse_grid(X_data, y_data, w0_arr, w1_arr):
    Z = np.zeros((len(w0_arr), len(w1_arr)))
    for i, w0 in enumerate(w0_arr):
        for j, w1 in enumerate(w1_arr):
            preds   = X_data @ np.array([w0, w1])
            Z[i, j] = ((preds - y_data) ** 2).mean()
    return Z

w0r = np.linspace(-3,  9, 50)
w1r = np.linspace(-5, 65, 50)
Z_raw = mse_grid(X_ep, y_epc, w0r, w1r)

w0s = np.linspace(-50, 50, 50)
w1s = np.linspace(-50, 50, 50)
Z_sc = mse_grid(X_ep_sc, y_epc, w0s, w1s)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

W0r, W1r = np.meshgrid(w0r, w1r)
cs = axes[0].contour(W0r, W1r, Z_raw.T, levels=25, cmap='RdYlGn_r')
axes[0].set_xlabel('weight for age (unscaled, range 20-70)')
axes[0].set_ylabel('weight for income (unscaled, range 0-1)')
axes[0].set_title('Unscaled: elongated valley -- GD zigzags')

W0s, W1s = np.meshgrid(w0s, w1s)
axes[1].contour(W0s, W1s, Z_sc.T, levels=25, cmap='RdYlGn_r')
axes[1].set_xlabel('weight for age (scaled)')
axes[1].set_ylabel('weight for income (scaled)')
axes[1].set_title('Scaled: round bowl -- GD goes straight to center')

plt.tight_layout()
plt.show()

print("Unscaled: age ranges 20-70, income 0-1 -- very different feature scales.")
print("The valley is narrow in the income direction, wide in the age direction.")
print("GD must use a tiny alpha to avoid overshooting income, wasting iterations on age.")
print()
print("After StandardScaler: both features have mean=0, std=1.")
print("The bowl is nearly circular -- GD can use a larger alpha and converges faster.")
print("Same training data, same model, different loss surface geometry.")


## Part 5 — Gradient Descent in Function Space: Gradient Boosted Trees

So far gradient descent has adjusted **weights** (numbers) to reduce error.

The key insight from Grokking Deep Learning Chapter 5:

> *Gradient descent is a general algorithm.  If you can calculate an error function and a delta,  
> gradient descent can show you how to move your weights to reduce your error.*

**Gradient Boosted Trees (GBT)** use this idea — but instead of adjusting weights, each step
**adds a new decision tree** that predicts the current residuals (= the negative gradient of the loss).

```
Step 0:  F₀ = mean(y)                      # predict the mean

Step 1:  residuals = y − F₀                # error signal  (= delta in the book)
         fit tree T₁ to residuals          # tree learns one gradient step
         F₁ = F₀ + α × T₁.predict(X)      # update predictions

Step 2:  residuals = y − F₁               # new error signal
         fit tree T₂ to residuals
         F₂ = F₁ + α × T₂.predict(X)

         ... repeat for N trees
```

Each tree is **one gradient descent step in function space**.  
Alpha (learning rate) is the same concept — too large → overfit in few rounds, too small → need more trees.

| GD on weights | GD on functions (GBT) |
|---|---|
| update: `w -= α × gradient` | update: `F += α × new_tree(X)` |
| gradient = `input × delta` | gradient = residuals `y − F` |
| alpha scales weight step | alpha scales tree contribution |
| diverge if alpha too large | overfit if alpha too large |


In [ ]:
np.random.seed(42)
n_b = 200
X_b = np.random.uniform(0, 10, (n_b, 3))
y_b = (2 * X_b[:, 0] ** 2
       + 3 * np.sin(X_b[:, 1])
       - X_b[:, 2]
       + np.random.normal(0, 1, n_b))

alpha_b  = 0.2
n_trees  = 60
max_dep  = 3

F        = np.full(n_b, y_b.mean())
mse_hist = []
res_hist = []
trees_b  = []

for t in range(n_trees):
    resid = y_b - F
    mse_hist.append(((y_b - F) ** 2).mean())
    if t in [0, 1, 4, 15, 40]:
        res_hist.append((t, resid.copy()))
    tree_b = DecisionTreeRegressor(max_depth=max_dep, random_state=t)
    tree_b.fit(X_b, resid)
    F += alpha_b * tree_b.predict(X_b)
    trees_b.append(tree_b)

mse_hist.append(((y_b - F) ** 2).mean())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(mse_hist, 'steelblue', lw=2)
axes[0].axhline(min(mse_hist), color='green', ls='--', lw=1,
                label='min MSE = {:.2f}'.format(min(mse_hist)))
axes[0].set_xlabel('boosting round (number of trees)')
axes[0].set_ylabel('MSE on training data')
axes[0].set_title('GBT loss curve -- same descending shape as weight-space GD')
axes[0].legend()

colors_b = plt.cm.Blues(np.linspace(0.3, 0.95, len(res_hist)))
for (rnd, res), col in zip(res_hist, colors_b):
    axes[1].hist(res, bins=20, alpha=0.5, color=col, label='round {}'.format(rnd))
axes[1].set_xlabel('residual value')
axes[1].set_ylabel('count')
axes[1].set_title('Residuals shrink each round (gradient signal reducing)')
axes[1].legend(fontsize=8)

axes[2].scatter(y_b, F, alpha=0.4, color='steelblue', s=20)
mn_b, mx_b = y_b.min(), y_b.max()
axes[2].plot([mn_b, mx_b], [mn_b, mx_b], 'r--', lw=1.5, label='perfect')
axes[2].set_xlabel('true y')
axes[2].set_ylabel('GBT prediction ({} trees)'.format(n_trees))
axes[2].set_title('GBT after {} trees -- R2={:.3f}'.format(n_trees, r2_score(y_b, F)))
axes[2].legend()

plt.tight_layout()
plt.show()

print("Each tree was trained to predict the current residuals -- what the model still gets wrong.")
print("Adding alpha * tree.predict(X) is exactly one gradient descent step.")
print("The loss curve descends the same way as weight-space GD from notebook 06.")
print()
print("Final training MSE after {} trees: {:.3f}".format(n_trees, mse_hist[-1]))
print("R2 after {} trees: {:.3f}".format(n_trees, r2_score(y_b, F)))


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_b, y_b, test_size=0.3, random_state=0)

models_cmp = {
    'Single Tree (depth 4)':          DecisionTreeRegressor(max_depth=4),
    'Random Forest (100 trees)':      RandomForestRegressor(n_estimators=100, random_state=0),
    'Gradient Boosting (100 trees)':  GradientBoostingRegressor(
                                          n_estimators=100, max_depth=3,
                                          learning_rate=0.1, random_state=0),
}

results_cmp = {}
for name, model in models_cmp.items():
    model.fit(X_tr, y_tr)
    train_r2 = r2_score(y_tr, model.predict(X_tr))
    test_r2  = r2_score(y_te, model.predict(X_te))
    results_cmp[name] = (train_r2, test_r2)

fig, ax = plt.subplots(figsize=(10, 4))
names_c   = list(results_cmp.keys())
train_r2s = [v[0] for v in results_cmp.values()]
test_r2s  = [v[1] for v in results_cmp.values()]
x_pos     = np.arange(len(names_c))

bars1 = ax.bar(x_pos - 0.18, train_r2s, 0.35, label='Train R2', color='steelblue')
bars2 = ax.bar(x_pos + 0.18, test_r2s,  0.35, label='Test R2',  color='#4CAF50')

for bar, val in zip(bars1, train_r2s):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            '{:.3f}'.format(val), ha='center', va='bottom', fontsize=9)
for bar, val in zip(bars2, test_r2s):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            '{:.3f}'.format(val), ha='center', va='bottom', fontsize=9)

ax.set_xticks(x_pos)
ax.set_xticklabels(names_c, fontsize=10)
ax.set_ylabel('R2 score')
ax.set_title('Model comparison -- GBT applies GD and outperforms plain trees')
ax.legend()
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print("Single tree:    high train R2, lower test R2 -- overfits to specific splits")
print("Random Forest:  averages many independent trees -- reduces variance, better test R2")
print("GBT:            each tree corrects the current gradient -- reduces bias sequentially")
print()
print("Key difference:")
print("  Random Forest  = parallel trees, averaged  (bagging)")
print("  GBT            = sequential trees, each one is a gradient descent step  (boosting)")


## Key Takeaways

### 1. The Derivative
- `weight_delta = input × delta` is the derivative of the error w.r.t. the weight
- It encodes both **direction** (up or down?) and **amount** (how big a step?)
- Moving opposite the derivative (`-= alpha * weight_delta`) walks down the loss bowl

### 2. Divergence and Alpha
- Large inputs make the loss bowl very steep → large gradient → easy to overshoot
- Alpha controls step size: too large → diverge, too small → slow convergence
- StandardScaler equalizes feature scales → round bowl → stable gradient descent

### 3. The Frozen Weight
- `weight_delta[i] = input[i] × delta = 0` whenever `input[i] = 0`
- That weight receives no update from that row — it is frozen by the data
- A feature that never fires can never have its weight learned

### 4. The Error Plane
- The shape of the loss surface is determined by training data, not the algorithm
- Unscaled features → elongated valley → GD zigzags
- Scaled features → round bowl → GD converges fast and stably

### 5. GD in Function Space (GBT)
| Concept | Neural Network GD | Gradient Boosted Trees |
|---|---|---|
| What is updated | weights | predictions (via new trees) |
| Step direction | −gradient w.r.t. weights | residuals = negative gradient of MSE |
| Learning rate α | scales weight update | scales each tree's contribution |
| Too large α | diverge | overfit in few rounds |
| Too small α | slow but stable | needs more trees, generalises better |

**The loss curve across boosting rounds and the loss curve across GD iterations are the same shape — because they are the same algorithm applied in different spaces.**
